# Tennis Statistics Preprocessing

## 1. Setup and locate the extracted statistics data

### 1.1 Define the dataset paths

In [1]:
from pathlib import Path
import polars as pl

MINI_PROJECT_ROOT = Path.cwd().parents[2]

DATA_ROOT = (
    MINI_PROJECT_ROOT
    / "Tennis Schema"
    / "tennis_data"
)

EXTRACT_ROOT = DATA_ROOT / "extracted"

### 1.2 Collect all statistics Parquet files

In [2]:
# Collect every extracted statistics Parquet file
# Each daily folder represents a separate snapshot of the available data

statistics_files = sorted(
    EXTRACT_ROOT.glob(
        "*/raw_statistics_parquet/*.parquet"
    )
)

print("Statistics file appearances:", len(statistics_files))

Statistics file appearances: 23291


## 2. Inspect the statistics data structure

### 2.1 Load one sample statistics file

In [3]:
# Read one statistics file as a sample
# Inspect the raw structure before making any cleaning decisions

sample_statistics_file = statistics_files[0]

statistics_sample = pl.read_parquet(
    sample_statistics_file
)

print("Sample file:", sample_statistics_file.name)
print("Shape:", statistics_sample.shape)

Sample file: statistics_11998445.parquet
Shape: (71, 13)


### 2.2 Inspect the sample schema

In [4]:
# Inspect the column names and data types in the sample statistics file

statistics_sample.schema

Schema([('match_id', Int64),
        ('period', String),
        ('statistic_category_name', String),
        ('statistic_name', String),
        ('home_stat', String),
        ('away_stat', String),
        ('compare_code', Int64),
        ('statistic_type', String),
        ('value_type', String),
        ('home_value', Int64),
        ('away_value', Int64),
        ('home_total', Float64),
        ('away_total', Float64)])

### 2.3 Preview the sample statistics rows

In [5]:
# Preview sample rows to understand how the statistics are stored

statistics_sample.head(20)

match_id,period,statistic_category_name,statistic_name,home_stat,away_stat,compare_code,statistic_type,value_type,home_value,away_value,home_total,away_total
i64,str,str,str,str,str,i64,str,str,i64,i64,f64,f64
11998445,"""ALL""","""service""","""aces""","""12""","""6""",1,"""positive""","""event""",12,6,null,null
11998445,"""ALL""","""service""","""double_faults""","""2""","""7""",2,"""negative""","""event""",2,7,null,null
11998445,"""ALL""","""service""","""first_serve""","""57/101 (56%)""","""53/90 (59%)""",2,"""positive""","""team""",57,53,101.0,90.0
11998445,"""ALL""","""service""","""second_serve""","""42/44 (95%)""","""30/37 (81%)""",1,"""positive""","""team""",42,30,44.0,37.0
11998445,"""ALL""","""service""","""first_serve_points""","""42/57 (74%)""","""39/53 (74%)""",1,"""positive""","""team""",42,39,57.0,53.0
…,…,…,…,…,…,…,…,…,…,…,…,…
11998445,"""ALL""","""return""","""first_serve_return_points""","""14/53 (26%)""","""15/57 (26%)""",2,"""positive""","""team""",14,15,53.0,57.0
11998445,"""ALL""","""return""","""second_serve_return_points""","""21/37 (56%)""","""26/44 (59%)""",2,"""positive""","""team""",21,26,37.0,44.0
11998445,"""ALL""","""return""","""return_games_played""","""16""","""16""",3,"""positive""","""event""",16,16,null,null


### 2.4 Inspect categorical values in the sample

In [6]:
for column in [
    "period",
    "statistic_category_name",
    "statistic_type",
    "value_type",
]:
    values = (
        statistics_sample
        .get_column(column)
        .unique()
        .sort()
        .to_list()
    )

    print(f"{column}: {values}")

period: ['1ST', '2ND', '3RD', 'ALL']
statistic_category_name: ['games', 'miscellaneous', 'points', 'return', 'service']
statistic_type: ['negative', 'positive']
value_type: ['event', 'team']


### 2.5 Inspect the statistic names in the sample

In [7]:
# Show the statistic names grouped by category

statistic_pairs = (
    statistics_sample
    .select([
        "statistic_category_name",
        "statistic_name",
    ])
    .unique()
    .sort([
        "statistic_category_name",
        "statistic_name",
    ])
)

for category in (
    statistic_pairs
    .get_column("statistic_category_name")
    .unique()
    .sort()
    .to_list()
):
    print(f"\n{category}:")

    names = (
        statistic_pairs
        .filter(
            pl.col("statistic_category_name") == category
        )
        .get_column("statistic_name")
        .to_list()
    )

    for name in names:
        print(" -", name)


games:
 - max_games_in_a_row
 - service_games_won
 - total_won

miscellaneous:
 - tiebreaks

points:
 - max_points_in_a_row
 - receiver_points_won
 - service_points_won
 - total

return:
 - break_points_converted
 - first_serve_return_points
 - return_games_played
 - second_serve_return_points

service:
 - aces
 - break_points_saved
 - double_faults
 - first_serve
 - first_serve_points
 - second_serve
 - second_serve_points
 - service_games_played


## 3. Investigate schema consistency and missing values

### 3.1 Check schema consistency across all statistics files

In [8]:
# Check whether every statistics file uses the same schema

statistics_schemas = {
    tuple(pl.read_parquet_schema(file).items())
    for file in statistics_files
}

print("Number of unique schemas:", len(statistics_schemas))

Number of unique schemas: 1


### 3.2 Check missing values in the sample file

In [9]:
# Count missing values in each column of the sample file

statistics_sample.null_count()

match_id,period,statistic_category_name,statistic_name,home_stat,away_stat,compare_code,statistic_type,value_type,home_value,away_value,home_total,away_total
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,43,43


### 3.3 Investigate missing totals by value type

In [10]:
# Check whether missing totals are associated with particular value types

(
    statistics_sample
    .group_by("value_type")
    .agg([
        pl.len().alias("rows"),
        pl.col("home_total").is_null().sum().alias("home_total_nulls"),
        pl.col("away_total").is_null().sum().alias("away_total_nulls"),
    ])
    .sort("value_type")
)

value_type,rows,home_total_nulls,away_total_nulls
str,u32,u32,u32
"""event""",43,43,43
"""team""",28,0,0


### 3.4 Check missing values across all statistics files

In [11]:
# Check missing values across all statistics files

statistics_all_lazy = pl.scan_parquet(statistics_files)

statistics_all_nulls = (
    statistics_all_lazy
    .select([
        pl.col(column).is_null().sum().alias(column)
        for column in statistics_sample.columns
    ])
    .collect()
)

statistics_all_nulls

match_id,period,statistic_category_name,statistic_name,home_stat,away_stat,compare_code,statistic_type,value_type,home_value,away_value,home_total,away_total
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,824739,824739


### 3.5 Verify the missing-total pattern across the full dataset

In [12]:
# Verify whether missing totals follow the same value_type pattern across the complete statistics dataset

(
    statistics_all_lazy
    .group_by("value_type")
    .agg([
        pl.len().alias("rows"),
        pl.col("home_total").is_null().sum().alias("home_total_nulls"),
        pl.col("away_total").is_null().sum().alias("away_total_nulls"),
    ])
    .sort("value_type")
    .collect()
)

value_type,rows,home_total_nulls,away_total_nulls
str,u32,u32,u32
"""event""",824739,824739,824739
"""team""",533495,0,0


## 4. Investigate repeated daily snapshots

### 4.1 Count unique and repeated statistics filenames

In [13]:
# Count how often each statistics filename appears across daily snapshots

from collections import Counter

statistics_filename_counts = Counter(
    file.name for file in statistics_files
)

unique_statistics_files = len(statistics_filename_counts)

repeated_statistics_files = sum(
    count > 1
    for count in statistics_filename_counts.values()
)

single_statistics_files = sum(
    count == 1
    for count in statistics_filename_counts.values()
)

print("File appearances:", len(statistics_files))
print("Unique filenames:", unique_statistics_files)
print("Repeated filenames:", repeated_statistics_files)
print("Filenames appearing once:", single_statistics_files)

File appearances: 23291
Unique filenames: 11393
Repeated filenames: 10636
Filenames appearing once: 757


### 4.2 Check whether repeated statistics snapshots changed

In [14]:
from collections import defaultdict

# Group all file paths by filename
statistics_paths_by_name = defaultdict(list)

for file in statistics_files:
    statistics_paths_by_name[file.name].append(file)

unchanged_repeated_statistics = []
changed_repeated_statistics = []

for filename, paths in statistics_paths_by_name.items():
    if len(paths) <= 1:
        continue

    # Sort rows so row order alone does not count as a data change
    first_df = pl.read_parquet(paths[0]).sort(statistics_sample.columns)

    changed = False

    for path in paths[1:]:
        current_df = pl.read_parquet(path).sort(statistics_sample.columns)

        if not first_df.equals(current_df):
            changed = True
            break

    if changed:
        changed_repeated_statistics.append(filename)
    else:
        unchanged_repeated_statistics.append(filename)

print("Repeated filenames:", repeated_statistics_files)
print("Unchanged repeated filenames:", len(unchanged_repeated_statistics))
print("Changed repeated filenames:", len(changed_repeated_statistics))

Repeated filenames: 10636
Unchanged repeated filenames: 9031
Changed repeated filenames: 1605


### 4.3 Check whether changed snapshots also changed row counts

In [15]:
# Compare row counts across the changed repeated statistics files

changed_row_count = []
same_row_count_but_changed = []

for filename in changed_repeated_statistics:
    paths = sorted(statistics_paths_by_name[filename])

    row_counts = [
        pl.read_parquet(path).height
        for path in paths
    ]

    if len(set(row_counts)) > 1:
        changed_row_count.append(filename)
    else:
        same_row_count_but_changed.append(filename)

print("Changed repeated filenames:", len(changed_repeated_statistics))
print("Changed row count:", len(changed_row_count))
print("Same row count but changed content:", len(same_row_count_but_changed))

Changed repeated filenames: 1605
Changed row count: 147
Same row count but changed content: 1458


### 4.4 Inspect an example of a same-size changed snapshot

In [16]:
# Pick one repeated file whose content changed even though the number of rows stayed the same

example_filename = same_row_count_but_changed[0]

example_paths = sorted(
    statistics_paths_by_name[example_filename]
)

print("Filename:", example_filename)

for path in example_paths:
    df = pl.read_parquet(path)

    print(
        path.parent.parent.name,
        "rows:",
        df.height,
    )

Filename: statistics_12058641.parquet
20240211 rows: 54
20240212 rows: 54


In [17]:
# Compare the two snapshots and show the rows whose contents changed

old_snapshot = pl.read_parquet(example_paths[0])
new_snapshot = pl.read_parquet(example_paths[1])

all_columns = old_snapshot.columns

old_only = (
    old_snapshot
    .join(
        new_snapshot,
        on=all_columns,
        how="anti",
    )
    .sort(["period", "statistic_category_name", "statistic_name"])
)

new_only = (
    new_snapshot
    .join(
        old_snapshot,
        on=all_columns,
        how="anti",
    )
    .sort(["period", "statistic_category_name", "statistic_name"])
)

print("Rows only in older snapshot:", old_only.height)
print("Rows only in newer snapshot:", new_only.height)

print("\nOlder version:")
print(old_only)

print("\nNewer version:")
print(new_only)

Rows only in older snapshot: 41
Rows only in newer snapshot: 41

Older version:
shape: (41, 13)
┌──────────┬────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ match_id ┆ period ┆ statistic_ ┆ statistic_ ┆ … ┆ home_value ┆ away_valu ┆ home_tota ┆ away_tota │
│ ---      ┆ ---    ┆ category_n ┆ name       ┆   ┆ ---        ┆ e         ┆ l         ┆ l         │
│ i64      ┆ str    ┆ ame        ┆ ---        ┆   ┆ i64        ┆ ---       ┆ ---       ┆ ---       │
│          ┆        ┆ ---        ┆ str        ┆   ┆            ┆ i64       ┆ f64       ┆ f64       │
│          ┆        ┆ str        ┆            ┆   ┆            ┆           ┆           ┆           │
╞══════════╪════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 12058641 ┆ 1ST    ┆ games      ┆ max_games_ ┆ … ┆ 0          ┆ 6         ┆ null      ┆ null      │
│          ┆        ┆            ┆ in_a_row   ┆   ┆            ┆           ┆           ┆        

### 4.5 Compare revised statistics side by side

In [18]:
# Match the same logical statistic across the two snapshots so revisions can be compared side by side

comparison_keys = [
    "match_id",
    "period",
    "statistic_category_name",
    "statistic_name",
]

snapshot_comparison = (
    old_snapshot
    .join(
        new_snapshot,
        on=comparison_keys,
        how="inner",
        suffix="_new",
    )
    .filter(
        (pl.col("home_stat") != pl.col("home_stat_new"))
        | (pl.col("away_stat") != pl.col("away_stat_new"))
        | (pl.col("home_value") != pl.col("home_value_new"))
        | (pl.col("away_value") != pl.col("away_value_new"))
        | (pl.col("home_total") != pl.col("home_total_new"))
        | (pl.col("away_total") != pl.col("away_total_new"))
    )
    .select([
        "period",
        "statistic_name",
        "home_stat",
        "home_stat_new",
        "away_stat",
        "away_stat_new",
    ])
    .sort(["period", "statistic_name"])
)

print("Revised statistics:", snapshot_comparison.height)
snapshot_comparison.head(20)

Revised statistics: 15


period,statistic_name,home_stat,home_stat_new,away_stat,away_stat_new
str,str,str,str,str,str
"""2ND""","""break_points_converted""","""2""","""1""","""3""","""3"""
"""2ND""","""break_points_saved""","""2/5 (40%)""","""2/5 (40%)""","""-1/1 (-100%)""","""0/1 (0%)"""
"""ALL""","""aces""","""2""","""0""","""4""","""6"""
"""ALL""","""break_points_converted""","""2""","""1""","""6""","""6"""
"""ALL""","""break_points_saved""","""3/9 (33%)""","""3/9 (33%)""","""0/2 (0%)""","""0/1 (0%)"""
…,…,…,…,…,…
"""ALL""","""second_serve""","""14/17 (82%)""","""16/19 (84%)""","""11/14 (79%)""","""14/15 (93%)"""
"""ALL""","""second_serve_points""","""6/18 (33%)""","""7/19 (37%)""","""6/13 (46%)""","""8/15 (53%)"""
"""ALL""","""second_serve_return_points""","""7/13 (53%)""","""7/15 (46%)""","""12/18 (66%)""","""12/19 (63%)"""


### 4.6 Investigate non-value changes in revised snapshots

In [19]:
# Check which metadata fields changed between the two snapshots

metadata_comparison = (
    old_snapshot
    .join(
        new_snapshot,
        on=comparison_keys,
        how="inner",
        suffix="_new",
    )
    .select([
        "period",
        "statistic_name",

        (pl.col("compare_code") != pl.col("compare_code_new"))
        .alias("compare_code_changed"),

        (pl.col("statistic_type") != pl.col("statistic_type_new"))
        .alias("statistic_type_changed"),

        (pl.col("value_type") != pl.col("value_type_new"))
        .alias("value_type_changed"),
    ])
    .filter(
        pl.col("compare_code_changed")
        | pl.col("statistic_type_changed")
        | pl.col("value_type_changed")
    )
)

print("Rows with metadata changes:", metadata_comparison.height)
metadata_comparison


Rows with metadata changes: 3


period,statistic_name,compare_code_changed,statistic_type_changed,value_type_changed
str,str,bool,bool,bool
"""ALL""","""double_faults""",true,false,false
"""ALL""","""second_serve""",true,false,false
"""2ND""","""break_points_saved""",true,false,false


### 4.7 Count the truly changed statistics in the example

In [20]:
# Compare the same logical statistic across snapshots and identify any value or metadata change

example_changes = (
    old_snapshot
    .join(
        new_snapshot,
        on=comparison_keys,
        how="inner",
        suffix="_new",
    )
    .with_columns([
        (
            (pl.col("home_stat") != pl.col("home_stat_new"))
            | (pl.col("away_stat") != pl.col("away_stat_new"))
            | (pl.col("home_value") != pl.col("home_value_new"))
            | (pl.col("away_value") != pl.col("away_value_new"))
            | (
                pl.col("home_total")
                .fill_null(float("-inf"))
                !=
                pl.col("home_total_new")
                .fill_null(float("-inf"))
            )
            | (
                pl.col("away_total")
                .fill_null(float("-inf"))
                !=
                pl.col("away_total_new")
                .fill_null(float("-inf"))
            )
            | (pl.col("compare_code") != pl.col("compare_code_new"))
            | (pl.col("statistic_type") != pl.col("statistic_type_new"))
            | (pl.col("value_type") != pl.col("value_type_new"))
        ).alias("changed")
    ])
    .filter(pl.col("changed"))
)

print("Truly changed statistics:", example_changes.height)

Truly changed statistics: 15


### 4.8 Characterize row-count changes across repeated snapshots

In [21]:
# Check whether repeated statistics files gained or lost rows over time

row_count_increase = []
row_count_decrease = []
row_count_both = []

for filename in changed_row_count:
    paths = sorted(statistics_paths_by_name[filename])

    row_counts = [
        pl.read_parquet(path).height
        for path in paths
    ]

    increases = any(
        current > previous
        for previous, current in zip(row_counts, row_counts[1:])
    )

    decreases = any(
        current < previous
        for previous, current in zip(row_counts, row_counts[1:])
    )

    if increases and decreases:
        row_count_both.append(filename)
    elif increases:
        row_count_increase.append(filename)
    elif decreases:
        row_count_decrease.append(filename)

print("Changed-row-count filenames:", len(changed_row_count))
print("Only increased:", len(row_count_increase))
print("Only decreased:", len(row_count_decrease))
print("Both increased and decreased:", len(row_count_both))

Changed-row-count filenames: 147
Only increased: 138
Only decreased: 9
Both increased and decreased: 0


### 4.9 Inspect snapshots whose row counts decreased

In [22]:
# Show row-count history for statistics files that lost rows

for filename in row_count_decrease:
    print(f"\n{filename}")

    for path in sorted(statistics_paths_by_name[filename]):
        df = pl.read_parquet(path)

        print(
            path.parent.parent.name,
            "rows:",
            df.height,
        )


statistics_12061015.parquet
20240211 rows: 52
20240212 rows: 51

statistics_12093076.parquet
20240222 rows: 34
20240223 rows: 30

statistics_12101983.parquet
20240225 rows: 50
20240226 rows: 49

statistics_12104736.parquet
20240226 rows: 49
20240227 rows: 47

statistics_12104738.parquet
20240226 rows: 50
20240227 rows: 49

statistics_12107071.parquet
20240226 rows: 52
20240227 rows: 51

statistics_12108467.parquet
20240227 rows: 52
20240228 rows: 51

statistics_12108547.parquet
20240227 rows: 51
20240228 rows: 50

statistics_12183733.parquet
20240322 rows: 51
20240323 rows: 50


### 4.10 Identify the statistics removed from later snapshots

In [23]:
# Identify logical statistics that existed in an older snapshot but disappeared from the following snapshot

statistic_keys = [
    "match_id",
    "period",
    "statistic_category_name",
    "statistic_name",
]

removed_statistics = []

for filename in row_count_decrease:
    paths = sorted(statistics_paths_by_name[filename])

    for old_path, new_path in zip(paths, paths[1:]):
        old_df = pl.read_parquet(old_path)
        new_df = pl.read_parquet(new_path)

        if new_df.height >= old_df.height:
            continue

        removed = (
            old_df
            .select(statistic_keys)
            .unique()
            .join(
                new_df.select(statistic_keys).unique(),
                on=statistic_keys,
                how="anti",
            )
        )

        if removed.height > 0:
            removed = removed.with_columns([
                pl.lit(filename).alias("filename"),
                pl.lit(old_path.parent.parent.name).alias("old_date"),
                pl.lit(new_path.parent.parent.name).alias("new_date"),
            ])

            removed_statistics.append(removed)

removed_statistics_df = pl.concat(removed_statistics)

removed_statistics_df.select([
    "filename",
    "old_date",
    "new_date",
    "period",
    "statistic_category_name",
    "statistic_name",
]).sort([
    "filename",
    "period",
    "statistic_name",
])

filename,old_date,new_date,period,statistic_category_name,statistic_name
str,str,str,str,str,str
"""statistics_12061015.parquet""","""20240211""","""20240212""","""ALL""","""miscellaneous""","""tiebreaks"""
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""games""","""max_games_in_a_row"""
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""service""","""second_serve"""
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""service""","""second_serve_points"""
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""miscellaneous""","""tiebreaks"""
…,…,…,…,…,…
"""statistics_12104738.parquet""","""20240226""","""20240227""","""ALL""","""games""","""max_games_in_a_row"""
"""statistics_12107071.parquet""","""20240226""","""20240227""","""ALL""","""miscellaneous""","""tiebreaks"""
"""statistics_12108467.parquet""","""20240227""","""20240228""","""ALL""","""miscellaneous""","""tiebreaks"""


### 4.11 Inspect the values of removed statistics

In [24]:
# Show the complete old rows that disappeared from later snapshots

removed_rows = []

for filename in row_count_decrease:
    paths = sorted(statistics_paths_by_name[filename])

    for old_path, new_path in zip(paths, paths[1:]):
        old_df = pl.read_parquet(old_path)
        new_df = pl.read_parquet(new_path)

        if new_df.height >= old_df.height:
            continue

        removed = (
            old_df
            .join(
                new_df.select(statistic_keys).unique(),
                on=statistic_keys,
                how="anti",
            )
            .with_columns([
                pl.lit(filename).alias("filename"),
                pl.lit(old_path.parent.parent.name).alias("old_date"),
                pl.lit(new_path.parent.parent.name).alias("new_date"),
            ])
        )

        if removed.height > 0:
            removed_rows.append(removed)

removed_rows_df = pl.concat(removed_rows)

removed_rows_df.select([
    "filename",
    "old_date",
    "new_date",
    "period",
    "statistic_name",
    "home_stat",
    "away_stat",
    "home_value",
    "away_value",
    "home_total",
    "away_total",
]).sort([
    "filename",
    "period",
    "statistic_name",
])

filename,old_date,new_date,period,statistic_name,home_stat,away_stat,home_value,away_value,home_total,away_total
str,str,str,str,str,str,str,i64,i64,f64,f64
"""statistics_12061015.parquet""","""20240211""","""20240212""","""ALL""","""tiebreaks""","""0""","""0""",0,0,null,null
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""max_games_in_a_row""","""0""","""3""",0,3,null,null
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""second_serve""","""4/5 (80%)""","""0/0 (0%)""",4,0,5.0,0.0
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""second_serve_points""","""1/5 (20%)""","""0/0 (0%)""",1,0,5.0,0.0
"""statistics_12093076.parquet""","""20240222""","""20240223""","""ALL""","""tiebreaks""","""0""","""0""",0,0,null,null
…,…,…,…,…,…,…,…,…,…,…
"""statistics_12104738.parquet""","""20240226""","""20240227""","""ALL""","""max_games_in_a_row""","""0""","""12""",0,12,null,null
"""statistics_12107071.parquet""","""20240226""","""20240227""","""ALL""","""tiebreaks""","""0""","""0""",0,0,null,null
"""statistics_12108467.parquet""","""20240227""","""20240228""","""ALL""","""tiebreaks""","""0""","""0""",0,0,null,null


### 4.12 Summarize the removed statistic types

In [25]:
(
    removed_rows_df
    .group_by("statistic_name")
    .agg(
        pl.len().alias("removed_rows")
    )
    .sort("removed_rows", descending=True)
)

statistic_name,removed_rows
str,u32
"""tiebreaks""",6
"""max_games_in_a_row""",5
"""second_serve_points""",1
"""second_serve""",1


## 5. Investigate statistic names, categories, periods, and value formats

### 5.1 Inspect all unique statistic names in the full dataset

In [26]:
all_statistic_names = (
    statistics_all_lazy
    .select("statistic_name")
    .unique()
    .sort("statistic_name")
    .collect()
)

print(
    all_statistic_names
    .get_column("statistic_name")
    .to_list()
)

['aces', 'break_points_converted', 'break_points_saved', 'double_faults', 'first_serve', 'first_serve_points', 'first_serve_return_points', 'max_games_in_a_row', 'max_points_in_a_row', 'receiver_points_won', 'return_games_played', 'second_serve', 'second_serve_points', 'second_serve_return_points', 'service_games_played', 'service_games_won', 'service_points_won', 'tiebreaks', 'total', 'total_won']


### 5.2 Map each statistic to its category and value type

In [27]:
statistic_metadata = (
    statistics_all_lazy
    .select([
        "statistic_category_name",
        "statistic_name",
        "statistic_type",
        "value_type",
    ])
    .unique()
    .sort([
        "statistic_category_name",
        "statistic_name",
    ])
    .collect()
)

for row in statistic_metadata.iter_rows(named=True):
    print(
        f"{row['statistic_category_name']:15} | "
        f"{row['statistic_name']:30} | "
        f"{row['statistic_type']:8} | "
        f"{row['value_type']}"
    )

games           | max_games_in_a_row             | positive | event
games           | service_games_won              | positive | event
games           | total_won                      | positive | event
miscellaneous   | tiebreaks                      | positive | event
points          | max_points_in_a_row            | positive | event
points          | receiver_points_won            | positive | event
points          | service_points_won             | positive | event
points          | total                          | positive | event
return          | break_points_converted         | positive | event
return          | first_serve_return_points      | positive | team
return          | return_games_played            | positive | event
return          | second_serve_return_points     | positive | team
service         | aces                           | positive | event
service         | break_points_saved             | positive | team
service         | double_faults                  | 

### 5.3 Inspect all period values in the full dataset

In [28]:
all_periods = (
    statistics_all_lazy
    .select("period")
    .unique()
    .sort("period")
    .collect()
    .get_column("period")
    .to_list()
)

print(all_periods)

['1ST', '2ND', '3RD', 'ALL']


### 5.4 Check how event and team values are formatted

In [29]:
value_format_check = (
    statistics_all_lazy
    .group_by("value_type")
    .agg([
        pl.len().alias("rows"),

        pl.col("home_stat")
        .str.contains("/")
        .sum()
        .alias("home_with_slash"),

        pl.col("home_stat")
        .str.contains("%")
        .sum()
        .alias("home_with_percent"),

        pl.col("away_stat")
        .str.contains("/")
        .sum()
        .alias("away_with_slash"),

        pl.col("away_stat")
        .str.contains("%")
        .sum()
        .alias("away_with_percent"),
    ])
    .sort("value_type")
    .collect()
)

value_format_check

value_type,rows,home_with_slash,home_with_percent,away_with_slash,away_with_percent
str,u32,u32,u32,u32,u32
"""event""",824739,0,0,0,0
"""team""",533495,533495,533495,533495,533495


## 6. Build the full statistics dataset

### 6.1 Combine all statistics files and add snapshot date

In [30]:
# Combine all statistics snapshots into one dataset and preserve the date each snapshot came from

statistics_all = (
    pl.read_parquet(
        statistics_files,
        include_file_paths="source_file",
    )
    .with_columns(
        pl.col("source_file")
        .str.extract(r"[\\/](\d{8})[\\/]raw_statistics_parquet", 1)
        .str.strptime(pl.Date, "%Y%m%d")
        .alias("snapshot_date")
    )
    .drop("source_file")
)

print("Combined shape:", statistics_all.shape)
print("Snapshot date range:",
      statistics_all["snapshot_date"].min(),
      "to",
      statistics_all["snapshot_date"].max())

Combined shape: (1358234, 14)
Snapshot date range: 2024-02-01 to 2024-03-31


## 7. Validate statistical consistency

### 7.1 Check for exact duplicate rows

In [31]:
# Check for completely identical rows

exact_duplicate_rows = (
    statistics_all
    .group_by(statistics_all.columns)
    .len()
    .filter(pl.col("len") > 1)
)

print("Exact duplicate groups:", exact_duplicate_rows.height)

Exact duplicate groups: 0


### 7.2 Check for duplicate statistics within the same snapshot

In [32]:
statistic_key = [
    "snapshot_date",
    "match_id",
    "period",
    "statistic_category_name",
    "statistic_name",
]

duplicate_statistic_keys = (
    statistics_all
    .group_by(statistic_key)
    .len()
    .filter(pl.col("len") > 1)
)

print("Duplicate statistic keys:", duplicate_statistic_keys.height)

Duplicate statistic keys: 0


### 7.3 Check for impossible negative values and invalid ratios

In [33]:
# Check for obviously invalid numeric statistics

negative_values = (
    statistics_all
    .filter(
        (pl.col("home_value") < 0)
        | (pl.col("away_value") < 0)
        | (pl.col("home_total") < 0)
        | (pl.col("away_total") < 0)
    )
)

invalid_team_ratios = (
    statistics_all
    .filter(
        (pl.col("value_type") == "team")
        & (
            (pl.col("home_value") > pl.col("home_total"))
            | (pl.col("away_value") > pl.col("away_total"))
        )
    )
)

print("Rows with negative values:", negative_values.height)
print("Rows with numerator greater than total:", invalid_team_ratios.height)

Rows with negative values: 30
Rows with numerator greater than total: 9459


### 7.4 Identify which statistics caused the invalid-value checks

In [34]:
print("Negative values by statistic:")

print(
    negative_values
    .group_by("statistic_name")
    .len()
    .sort("len", descending=True)
)

print("\nValue greater than total by statistic:")

print(
    invalid_team_ratios
    .group_by("statistic_name")
    .len()
    .sort("len", descending=True)
)

Negative values by statistic:
shape: (3, 2)
┌────────────────────────────┬─────┐
│ statistic_name             ┆ len │
│ ---                        ┆ --- │
│ str                        ┆ u32 │
╞════════════════════════════╪═════╡
│ second_serve_return_points ┆ 16  │
│ break_points_saved         ┆ 11  │
│ first_serve_return_points  ┆ 3   │
└────────────────────────────┴─────┘

Value greater than total by statistic:
shape: (7, 2)
┌────────────────────────────┬──────┐
│ statistic_name             ┆ len  │
│ ---                        ┆ ---  │
│ str                        ┆ u32  │
╞════════════════════════════╪══════╡
│ second_serve               ┆ 3570 │
│ first_serve_points         ┆ 1981 │
│ first_serve                ┆ 1923 │
│ second_serve_return_points ┆ 850  │
│ second_serve_points        ┆ 507  │
│ break_points_saved         ┆ 450  │
│ first_serve_return_points  ┆ 178  │
└────────────────────────────┴──────┘


### 7.5 Inspect examples of suspicious team statistics

In [35]:
# Inspect a small sample of rows where the numerator is greater than the total

(
    invalid_team_ratios
    .select([
        "snapshot_date",
        "match_id",
        "period",
        "statistic_name",
        "home_stat",
        "away_stat",
        "home_value",
        "away_value",
        "home_total",
        "away_total",
    ])
    .sort(["statistic_name", "snapshot_date"])
    .head(20)
)

snapshot_date,match_id,period,statistic_name,home_stat,away_stat,home_value,away_value,home_total,away_total
date,i64,str,str,str,str,i64,i64,f64,f64
2024-03-01,12115294,"""3RD""","""break_points_saved""","""2/4 (50%)""","""3/3 (100%)""",2,4,4.0,3.0
2024-03-01,12088086,"""2ND""","""break_points_saved""","""5/5 (100%)""","""0/1 (0%)""",8,0,5.0,1.0
2024-03-01,12088086,"""ALL""","""break_points_saved""","""5/6 (83%)""","""4/7 (57%)""",8,2,6.0,7.0
2024-03-01,12112743,"""ALL""","""break_points_saved""","""5/8 (62%)""","""7/8 (87%)""",0,10,8.0,8.0
2024-03-01,12088076,"""2ND""","""break_points_saved""","""5/7 (71%)""","""2/4 (50%)""",8,0,7.0,4.0
…,…,…,…,…,…,…,…,…,…
2024-03-01,12112897,"""ALL""","""break_points_saved""","""4/9 (44%)""","""4/4 (100%)""",2,6,9.0,4.0
2024-03-01,12088056,"""1ST""","""break_points_saved""","""6/7 (85%)""","""2/2 (100%)""",8,2,7.0,2.0
2024-03-01,12116852,"""2ND""","""break_points_saved""","""5/7 (71%)""","""6/7 (85%)""",0,8,7.0,7.0


### 7.6 Distinguish numeric-column mismatches from invalid displayed statistics

In [36]:
# Parse the numerator and denominator directly from the displayed team statistics

team_check = (
    statistics_all
    .filter(pl.col("value_type") == "team")
    .with_columns([
        pl.col("home_stat")
        .str.extract(r"^(-?\d+)/", 1)
        .cast(pl.Int64)
        .alias("home_display_value"),

        pl.col("home_stat")
        .str.extract(r"/(-?\d+)", 1)
        .cast(pl.Int64)
        .alias("home_display_total"),

        pl.col("away_stat")
        .str.extract(r"^(-?\d+)/", 1)
        .cast(pl.Int64)
        .alias("away_display_value"),

        pl.col("away_stat")
        .str.extract(r"/(-?\d+)", 1)
        .cast(pl.Int64)
        .alias("away_display_total"),
    ])
)

numeric_mismatch = team_check.filter(
    (pl.col("home_value") != pl.col("home_display_value"))
    | (pl.col("away_value") != pl.col("away_display_value"))
    | (pl.col("home_total") != pl.col("home_display_total"))
    | (pl.col("away_total") != pl.col("away_display_total"))
)

invalid_displayed_stats = team_check.filter(
    (pl.col("home_display_value") < 0)
    | (pl.col("away_display_value") < 0)
    | (pl.col("home_display_total") < 0)
    | (pl.col("away_display_total") < 0)
    | (pl.col("home_display_value") > pl.col("home_display_total"))
    | (pl.col("away_display_value") > pl.col("away_display_total"))
)

print("Rows where numeric columns disagree with displayed stats:",
      numeric_mismatch.height)

print("Rows with actually invalid displayed ratios:",
      invalid_displayed_stats.height)

Rows where numeric columns disagree with displayed stats: 44203
Rows with actually invalid displayed ratios: 49


### 7.7 Create cleaned numeric values for team statistics

In [37]:
statistics_all = (
    statistics_all
    .with_columns([
        pl.when(pl.col("value_type") == "team")
        .then(
            pl.col("home_stat")
            .str.extract(r"^(-?\d+)/", 1)
            .cast(pl.Int64)
        )
        .otherwise(pl.col("home_value"))
        .alias("home_value_clean"),

        pl.when(pl.col("value_type") == "team")
        .then(
            pl.col("away_stat")
            .str.extract(r"^(-?\d+)/", 1)
            .cast(pl.Int64)
        )
        .otherwise(pl.col("away_value"))
        .alias("away_value_clean"),

        pl.when(pl.col("value_type") == "team")
        .then(
            pl.col("home_stat")
            .str.extract(r"/(-?\d+)", 1)
            .cast(pl.Float64)
        )
        .otherwise(pl.col("home_total"))
        .alias("home_total_clean"),

        pl.when(pl.col("value_type") == "team")
        .then(
            pl.col("away_stat")
            .str.extract(r"/(-?\d+)", 1)
            .cast(pl.Float64)
        )
        .otherwise(pl.col("away_total"))
        .alias("away_total_clean"),
    ])
)

In [38]:
statistics_all = (
    statistics_all
    .with_columns(
        (
            (pl.col("value_type") == "team")
            & (
                (pl.col("home_value_clean") < 0)
                | (pl.col("away_value_clean") < 0)
                | (pl.col("home_total_clean") < 0)
                | (pl.col("away_total_clean") < 0)
                | (pl.col("home_value_clean") > pl.col("home_total_clean"))
                | (pl.col("away_value_clean") > pl.col("away_total_clean"))
            )
        ).alias("invalid_team_stat")
    )
)

print(
    "Flagged invalid team-stat rows:",
    statistics_all["invalid_team_stat"].sum()
)

Flagged invalid team-stat rows: 49


In [40]:
print("Shape:", statistics_all.shape)

for column in statistics_all.columns:
    print(column)

Shape: (1358234, 19)
match_id
period
statistic_category_name
statistic_name
home_stat
away_stat
compare_code
statistic_type
value_type
home_value
away_value
home_total
away_total
snapshot_date
home_value_clean
away_value_clean
home_total_clean
away_total_clean
invalid_team_stat


### 7.8 Verify the invalid team-stat flag

In [41]:
print(
    "Flagged invalid team-stat rows:",
    statistics_all["invalid_team_stat"].sum()
)

Flagged invalid team-stat rows: 49


### 7.9 Replace unreliable numeric fields and remove helper columns

In [42]:
# Replace the original numeric fields with the validated clean versions

statistics_all = (
    statistics_all
    .with_columns([
        pl.col("home_value_clean").alias("home_value"),
        pl.col("away_value_clean").alias("away_value"),
        pl.col("home_total_clean").alias("home_total"),
        pl.col("away_total_clean").alias("away_total"),
    ])
    .drop([
        "home_value_clean",
        "away_value_clean",
        "home_total_clean",
        "away_total_clean",
    ])
)

print("Final working shape:", statistics_all.shape)

Final working shape: (1358234, 15)


## 8. Clean and validate the final dataset

### 8.1 Verify the final dataset shape and key checks

In [43]:
print("Final shape:", statistics_all.shape)
print("Duplicate statistic keys:", duplicate_statistic_keys.height)
print("Invalid team-stat rows:", statistics_all["invalid_team_stat"].sum())
print("Snapshot dates:",
      statistics_all["snapshot_date"].min(),
      "to",
      statistics_all["snapshot_date"].max())

Final shape: (1358234, 15)
Duplicate statistic keys: 0
Invalid team-stat rows: 49
Snapshot dates: 2024-02-01 to 2024-03-31


### 8.2 Null invalid numeric team statistics while preserving the rows

In [44]:
statistics_all = (
    statistics_all
    .with_columns([
        pl.when(
            (pl.col("value_type") == "team")
            & (
                (pl.col("home_value") < 0)
                | (pl.col("home_total") < 0)
                | (pl.col("home_value") > pl.col("home_total"))
            )
        )
        .then(None)
        .otherwise(pl.col("home_value"))
        .alias("home_value"),

        pl.when(
            (pl.col("value_type") == "team")
            & (
                (pl.col("home_value") < 0)
                | (pl.col("home_total") < 0)
                | (pl.col("home_value") > pl.col("home_total"))
            )
        )
        .then(None)
        .otherwise(pl.col("home_total"))
        .alias("home_total"),

        pl.when(
            (pl.col("value_type") == "team")
            & (
                (pl.col("away_value") < 0)
                | (pl.col("away_total") < 0)
                | (pl.col("away_value") > pl.col("away_total"))
            )
        )
        .then(None)
        .otherwise(pl.col("away_value"))
        .alias("away_value"),

        pl.when(
            (pl.col("value_type") == "team")
            & (
                (pl.col("away_value") < 0)
                | (pl.col("away_total") < 0)
                | (pl.col("away_value") > pl.col("away_total"))
            )
        )
        .then(None)
        .otherwise(pl.col("away_total"))
        .alias("away_total"),
    ])
)

print("Shape:", statistics_all.shape)
print("Flagged invalid rows:", statistics_all["invalid_team_stat"].sum())

Shape: (1358234, 15)
Flagged invalid rows: 49


### 8.3 Confirm that invalid numeric ratios are no longer usable

In [45]:
remaining_invalid_team_values = (
    statistics_all
    .filter(
        (pl.col("value_type") == "team")
        & (
            (pl.col("home_value") < 0)
            | (pl.col("away_value") < 0)
            | (pl.col("home_total") < 0)
            | (pl.col("away_total") < 0)
            | (pl.col("home_value") > pl.col("home_total"))
            | (pl.col("away_value") > pl.col("away_total"))
        )
    )
)

print(
    "Remaining invalid numeric team rows:",
    remaining_invalid_team_values.height
)

Remaining invalid numeric team rows: 0


## 9. Save and summarize the cleaned data

### 9.1 Save the cleaned statistics dataset

In [46]:
PROCESSED_ROOT = DATA_ROOT / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

statistics_output_path = (
    PROCESSED_ROOT / "statistics_clean.parquet"
)

statistics_all.write_parquet(
    statistics_output_path
)

print("Saved to:", statistics_output_path)

Saved to: d:\Learning\Daneshkar\Statistics\Stat with Python\Mini Project\Tennis Schema\tennis_data\processed\statistics_clean.parquet


### 9.2 Verify the saved dataset

In [47]:
saved_statistics = pl.read_parquet(
    statistics_output_path
)

print("Saved shape:", saved_statistics.shape)
print(
    "Exact saved-data match:",
    saved_statistics.equals(statistics_all)
)

Saved shape: (1358234, 15)
Exact saved-data match: True


### 9.3 Preprocessing summary

The statistics dataset was inspected, validated, cleaned, and saved while preserving all daily snapshot history.

- 23,291 statistics file appearances were discovered.
- 11,393 unique filenames were present.
- 10,636 filenames appeared on multiple daily snapshots.
- 9,031 repeated filenames remained unchanged.
- 1,605 repeated filenames changed across snapshots.
- Among changed snapshots, 147 changed row count:
  - 138 increased
  - 9 decreased
- Decreased snapshots were inspected and treated as provider revisions rather than deleted or reconstructed.

- All 23,291 files shared one consistent 13-column schema.
- Original missing values occurred only in `home_total` and `away_total`.
- These nulls were structural:
  - `event` statistics: 824,739 rows, totals always null.
  - `team` statistics: 533,495 rows, totals always present.

- The dataset contained 20 unique statistic names.
- Statistics were grouped into:
  - games
  - miscellaneous
  - points
  - return
  - service
- Available periods were:
  - `1ST`
  - `2ND`
  - `3RD`
  - `ALL`

- `event` statistics were consistently stored as simple counts.
- `team` statistics were consistently stored as ratio/percentage strings such as `57/101 (56%)`.

- The full combined dataset contained 1,358,234 rows.
- `snapshot_date` was added to preserve daily historical observations.
- Snapshot dates ranged from 2024-02-01 to 2024-03-31.

- No exact duplicate rows were found.
- No duplicate logical statistic keys were found within a snapshot.

- 44,203 team rows had at least one numeric value or total field that disagreed with the numerator or denominator stored in `home_stat` / `away_stat`.
- For `team` statistics, cleaned numeric values and totals were therefore reconstructed from the displayed ratio strings.
- Original `event` numeric values were retained.

- After checking displayed ratios for negative values and numerators greater than their denominators, 49 rows contained invalid team statistics.
- These rows were preserved and flagged with `invalid_team_stat = True`.
- Invalid numeric sides were set to null rather than deleting the observations.
- After cleaning, 0 invalid numeric team ratios remained.

Final cleaned dataset:

- Rows: 1,358,234
- Columns: 15
- Duplicate statistic keys: 0
- Flagged invalid team-stat rows: 49
- No rows were deleted.
- Daily snapshot history was preserved.

The cleaned dataset was saved as:

`processed/statistics_clean.parquet`

The saved Parquet file was reloaded and matched the in-memory cleaned dataset exactly.